# Ablation: Cross-Lingual BERTScore Comparison

Compares the **best configuration** for each language (IT and EN) using a
**unified BERTScore model** (`microsoft/mdeberta-v3-base`) for fair cross-lingual
evaluation.

**Motivation**: Default BERTScore uses `roberta-large` for EN and
`bert-base-multilingual-cased` for IT, making absolute scores incomparable.

**Memory strategy**: SigExt on CPU -> unloaded -> single LLM for both languages.

In [ ]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!uv pip install -e ../..
from huggingface_hub import login
login()

In [ ]:
from sm_sip.config import SigExtConfig
from sm_sip.data import get_test_data
from sm_sip.models import (
    load_sigext_model, unload_sigext_model,
    load_llm, create_summary_chain, preprocess_dataset,
)
from sm_sip.prompts import get_summary_prompt
from sm_sip.pipelines import run_inference, run_evaluation
from sm_sip.utils.io import save_results
from sm_sip.utils.gpu import clear_gpu_memory
import numpy as np

## Configuration

Best configs from previous ablation studies:

| Parameter | Italian (WITS) | English (ArXiv) |
|---|---|---|
| SigExt | `xlmr-5k-060t` | `xlmr-5k-060t` |
| Prompt | `source_aware` | `source_aware` |
| Quantization | `4bit` | `8bit` |
| LLM | Llama-3.1-8B | Llama-3.1-8B |

In [ ]:
UNIFIED_BERT_MODEL = 'microsoft/mdeberta-v3-base'

CONFIGS = {
    'it': {
        'sigext_preset': 'xlmr-5k-060t',
        'quantization': '4bit',
        'prompt_type': 'source_aware',
        'num_samples': 50,
    },
    'en': {
        'sigext_preset': 'xlmr-5k-060t',
        'quantization': '8bit',
        'prompt_type': 'source_aware',
        'num_samples': 50,
    },
}

## Run Inference + Evaluation

In [ ]:
results = {}

for lang, cfg in CONFIGS.items():
    print(f'\n{"="*60}')
    print(f'  Processing: {lang.upper()}')
    print(f'{"="*60}')

    # 1. Load SigExt on CPU, preprocess, unload
    sc = SigExtConfig.from_preset(lang, cfg['sigext_preset'])
    data = get_test_data(lang=lang, num_samples=cfg['num_samples'], skip_samples=sc.skip_samples)
    sm, st = load_sigext_model(sc.model_id, device='cpu')
    proc = preprocess_dataset(data, sm, st, lang=lang)
    unload_sigext_model(sm, st)

    # 2. Load LLM with best quantization for this language
    _, _, pipe = load_llm('meta-llama/Llama-3.1-8B-Instruct', cfg['quantization'])
    chain = create_summary_chain(pipe, get_summary_prompt(lang, cfg['prompt_type']))

    # 3. Run inference
    inf_res = run_inference(proc, chain)

    # 4a. Metrics with DEFAULT BERTScore model (original behavior)
    print(f'  Computing metrics with DEFAULT BERTScore model ({lang})...')
    metrics_default = run_evaluation(inf_res, lang=lang)

    # 4b. Metrics with UNIFIED BERTScore model (cross-lingual comparison)
    print(f'  Computing metrics with UNIFIED BERTScore model ({UNIFIED_BERT_MODEL})...')
    metrics_unified = run_evaluation(inf_res, lang=lang, bert_model_type=UNIFIED_BERT_MODEL)

    results[lang] = {
        'config': cfg,
        'num_samples': len(inf_res),
        'default_bert': metrics_default,
        'unified_bert': metrics_unified,
    }

    clear_gpu_memory()
    print(f'  Done: {lang.upper()}')

## Results Comparison

In [ ]:
print('\n' + '='*80)
print('  CROSS-LINGUAL COMPARISON: Default vs Unified BERTScore')
print('='*80)
print(f'{"Metric":<25} {"Italian":>12} {"English":>12} {"Gap (EN-IT)":>12}')
print('-'*65)

for label, key_d, key_u in [
    ('BERTScore (default)',   'default_bert', None),
    ('BERTScore (mdeberta)',  'unified_bert', None),
    ('ROUGE-1',              'default_bert', None),
    ('ROUGE-L',              'default_bert', None),
    ('KIR',                  'default_bert', None),
]:
    if 'BERTScore' in label:
        src = key_d
        it_val = results['it'][src]['bert_score']['mean']
        en_val = results['en'][src]['bert_score']['mean']
    elif 'ROUGE-1' in label:
        it_val = results['it']['default_bert']['rouge1']['mean']
        en_val = results['en']['default_bert']['rouge1']['mean']
    elif 'ROUGE-L' in label:
        it_val = results['it']['default_bert']['rougeL']['mean']
        en_val = results['en']['default_bert']['rougeL']['mean']
    elif 'KIR' in label:
        it_val = results['it']['default_bert']['kir']['mean']
        en_val = results['en']['default_bert']['kir']['mean']
    gap = en_val - it_val
    print(f'{label:<25} {it_val:>12.4f} {en_val:>12.4f} {gap:>+12.4f}')

print('-'*65)
print()
print('Key insight: compare the GAP for BERTScore (default) vs BERTScore (mdeberta).')
print('If the gap shrinks significantly with mdeberta, the original gap was due to model disparity.')

In [ ]:
save_results(
    {
        'ablation': 'cross_lingual',
        'unified_bert_model': UNIFIED_BERT_MODEL,
        'results': results,
    },
    'results/ablation_cross_lingual.json',
)
print('Results saved to results/ablation_cross_lingual.json')